# Mô hình 2: XGBoost Regressor

XGBoost được chọn vì dữ liệu có nhiều quan hệ phi tuyến giữa khí tượng, pollutant,
mùa vụ và các đặc trưng trễ PM2.5. Mô hình cây tăng cường cũng ít nhạy với thang đo đầu vào,
nên phù hợp cho dữ liệu dạng bảng sau feature engineering.

## Bài toán học máy

Mục tiêu là dự báo nồng độ bụi mịn `PM2.5` sau 24 giờ cho Hà Nội, TP.HCM và Đà Nẵng
dựa trên dữ liệu chất lượng không khí, khí tượng, thời gian, đặc trưng trễ và trung bình trượt.

Vì đây là chuỗi thời gian, dữ liệu được chia tuần tự theo `target_time`:
Train 70%, Validation 15%, Test 15%. Validation dùng để theo dõi/tinh chỉnh mô hình;
Test chỉ dùng để đánh giá cuối cùng.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

HORIZON = 24
PM25_BINS = [-np.inf, 12.0, 35.4, 55.4, 150.4, 250.4, np.inf]
PM25_LABELS = ["Good", "Moderate", "USG", "Unhealthy", "Very Unhealthy", "Hazardous"]

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def pm25_category(values):
    return pd.cut(values, bins=PM25_BINS, labels=PM25_LABELS, include_lowest=True)

## Chuẩn bị dữ liệu, mã hóa và split 70/15/15

In [ ]:
df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values(["city", "datetime"]).reset_index(drop=True)
df["target_pm25_24h"] = df.groupby("city")["pm25"].shift(-HORIZON)
df["target_time"] = df.groupby("city")["datetime"].shift(-HORIZON)

numeric_features = [
    "pm25", "pm10", "o3", "no2", "so2", "co",
    "temp", "humidity", "wind_speed", "wind_dir", "precip", "pressure", "cloud_cover",
    "factories_2km", "factories_5km",
    "hour", "day_of_week", "month", "is_weekend", "day_of_year",
    "pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h", "pm25_lag_12h", "pm25_lag_24h",
    "pm25_roll_6h", "pm25_roll_24h", "pm25_roll_72h",
]
categorical_features = ["city", "season"]
required = numeric_features + categorical_features + ["target_pm25_24h", "target_time"]
df_model = df.dropna(subset=required).copy()

encoded = pd.get_dummies(
    df_model[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=False,
    dtype=float,
)
feature_cols = encoded.columns.tolist()
model_df = pd.concat(
    [df_model[["datetime", "target_time", "city", "target_pm25_24h"]], encoded],
    axis=1,
)

unique_times = pd.Series(model_df["target_time"].sort_values().unique())
train_cut = unique_times.iloc[int(len(unique_times) * 0.70)]
val_cut = unique_times.iloc[int(len(unique_times) * 0.85)]

train_mask = model_df["target_time"] < train_cut
val_mask = (model_df["target_time"] >= train_cut) & (model_df["target_time"] < val_cut)
test_mask = model_df["target_time"] >= val_cut

X_train = model_df.loc[train_mask, feature_cols]
y_train = model_df.loc[train_mask, "target_pm25_24h"]
X_val = model_df.loc[val_mask, feature_cols]
y_val = model_df.loc[val_mask, "target_pm25_24h"]
X_test = model_df.loc[test_mask, feature_cols]
y_test = model_df.loc[test_mask, "target_pm25_24h"]
meta_test = df_model.loc[test_mask, ["datetime", "target_time", "city", "temp", "humidity", "wind_speed", "precip"]].copy()

split_table = pd.DataFrame(
    {
        "Tập dữ liệu": ["Train", "Validation", "Test"],
        "Số mẫu": [len(X_train), len(X_val), len(X_test)],
        "Tỷ lệ (%)": [
            len(X_train) / len(model_df) * 100,
            len(X_val) / len(model_df) * 100,
            len(X_test) / len(model_df) * 100,
        ],
        "Khoảng target_time": [
            f"< {train_cut}",
            f"{train_cut} -> < {val_cut}",
            f">= {val_cut}",
        ],
    }
)

print("Bảng 1. Tỷ lệ chia dữ liệu cho XGBoost")
display(split_table)
print("Số đặc trưng sau mã hóa:", len(feature_cols))

## Cấu hình huấn luyện

- Loss/objective: `reg:squarederror`, đánh giá bằng RMSE.
- Hyperparameters: `n_estimators=500`, `max_depth=6`, `learning_rate=0.05`,
  `subsample=0.8`, `colsample_bytree=0.8`, `min_child_weight=3`.
- Phương pháp tinh chỉnh: thử nghiệm thủ công có kiểm soát trên Validation.
- Early stopping: theo dõi Validation RMSE, không sử dụng Test để dừng sớm.

In [ ]:
xgb_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "random_state": 42,
    "n_jobs": 1,
    "tree_method": "hist",
}

try:
    model_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=30)
    model_xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50,
    )
except TypeError:
    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        early_stopping_rounds=30,
        verbose=50,
    )

train_pred = model_xgb.predict(X_train)
val_pred = model_xgb.predict(X_val)
test_pred = model_xgb.predict(X_test)

metrics = pd.DataFrame(
    [
        {
            "model": "XGBoost",
            "scope": "All cities",
            "train_rmse_ug_m3": rmse(y_train, train_pred),
            "val_rmse_ug_m3": rmse(y_val, val_pred),
            "test_rmse_ug_m3": rmse(y_test, test_pred),
            "test_mae_ug_m3": float(mean_absolute_error(y_test, test_pred)),
            "test_r2": float(r2_score(y_test, test_pred)),
        }
    ]
)
metrics.to_csv(RESULTS_DIR / "xgboost_metrics.csv", index=False, encoding="utf-8-sig")
print("Bảng 2. Chỉ số đánh giá XGBoost, đơn vị lỗi: µg/m³")
display(metrics)

## Learning curve trên Train và Validation

In [ ]:
eval_result = model_xgb.evals_result()
curve = pd.DataFrame(
    {
        "round": np.arange(1, len(eval_result["validation_0"]["rmse"]) + 1),
        "train_rmse": eval_result["validation_0"]["rmse"],
        "val_rmse": eval_result["validation_1"]["rmse"],
    }
)
curve.to_csv(RESULTS_DIR / "xgboost_learning_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["round"], curve["train_rmse"], label="Train RMSE")
ax.plot(curve["round"], curve["val_rmse"], label="Validation RMSE")
ax.set_title("XGBoost learning curve")
ax.set_xlabel("Boosting round")
ax.set_ylabel("RMSE (µg/m³)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgboost_learning_curve.png", dpi=180)
plt.show()

Hình 1. Learning curve của XGBoost. Mô hình được xem là hội tụ khi Validation RMSE không còn giảm đáng kể;
early stopping giúp hạn chế overfitting khi Train RMSE tiếp tục giảm nhưng Validation RMSE đi ngang.

## Confusion matrix sau khi phân nhóm PM2.5

Target huấn luyện của XGBoost là giá trị liên tục `target_pm25_24h`.
Confusion matrix dưới đây không biến bài toán thành classification; nó chỉ ánh xạ
`PM2.5` thực tế và `PM2.5` dự báo sang các nhóm rủi ro để phân tích cảnh báo.

In [ ]:
actual_cat = pm25_category(y_test)
pred_cat = pm25_category(np.clip(test_pred, 0, None))

cm = confusion_matrix(actual_cat, pred_cat, labels=PM25_LABELS)
cm_df = pd.DataFrame(cm, index=PM25_LABELS, columns=PM25_LABELS)
cm_df.to_csv(RESULTS_DIR / "xgboost_confusion_matrix.csv", encoding="utf-8-sig")

class_summary = pd.DataFrame(
    [
        {
            "model": "XGBoost",
            "accuracy": float(accuracy_score(actual_cat, pred_cat)),
            "precision_macro": float(precision_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "recall_macro": float(recall_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "f1_macro": float(f1_score(actual_cat, pred_cat, labels=PM25_LABELS, average="macro", zero_division=0)),
            "precision_weighted": float(precision_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
            "recall_weighted": float(recall_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
            "f1_weighted": float(f1_score(actual_cat, pred_cat, labels=PM25_LABELS, average="weighted", zero_division=0)),
        }
    ]
)
class_report = pd.DataFrame(
    classification_report(actual_cat, pred_cat, labels=PM25_LABELS, output_dict=True, zero_division=0)
).T
class_summary.to_csv(RESULTS_DIR / "xgboost_classification_summary.csv", index=False, encoding="utf-8-sig")
class_report.to_csv(RESULTS_DIR / "xgboost_classification_report.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("XGBoost confusion matrix by PM2.5 category")
ax.set_xlabel("Predicted category")
ax.set_ylabel("Actual category")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgboost_confusion_matrix.png", dpi=180)
plt.show()

print("Bảng 3. Chỉ số cảnh báo sau khi phân nhóm PM2.5")
display(class_summary)
print("Bảng 4. Ma trận nhầm lẫn XGBoost sau khi ánh xạ PM2.5")
display(cm_df)

Hình 2. Ma trận nhầm lẫn của XGBoost sau khi ánh xạ PM2.5 thực tế/dự báo sang nhóm rủi ro.

## Phân tích trường hợp sai

In [ ]:
error_df = meta_test.copy()
error_df["actual_pm25"] = y_test.to_numpy()
error_df["predicted_pm25"] = test_pred
error_df["abs_error_ug_m3"] = np.abs(error_df["actual_pm25"] - error_df["predicted_pm25"])
error_df["actual_category"] = actual_cat.astype(str)
error_df["predicted_category"] = pred_cat.astype(str)

def hypothesis(row):
    if row["predicted_pm25"] < row["actual_pm25"] and row["abs_error_ug_m3"] >= 30:
        return "Đánh giá thấp đỉnh ô nhiễm; có thể do sự kiện phát thải đột ngột hoặc thiếu biến nguồn thải."
    if row["precip"] > 1:
        return "Thời tiết chuyển trạng thái; mưa có thể làm PM2.5 giảm nhanh."
    if row["wind_speed"] > 15:
        return "Gió mạnh làm cơ chế khuếch tán khác với mẫu học thông thường."
    return "Vùng chuyển tiếp khó dự báo; đặc trưng trễ chưa mô tả đủ diễn biến tương lai."

error_df["error_hypothesis"] = error_df.apply(hypothesis, axis=1)
top_errors = error_df.sort_values("abs_error_ug_m3", ascending=False).head(20)
top_errors.to_csv(RESULTS_DIR / "xgboost_top_error_cases.csv", index=False, encoding="utf-8-sig")

print("Bảng 5. Các trường hợp sai lớn nhất của XGBoost, đơn vị lỗi: µg/m³")
display(top_errors.head(10))

## Feature importance

In [ ]:
importance = pd.Series(model_xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(20).to_csv(RESULTS_DIR / "xgboost_feature_importance.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 6))
importance.head(15).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 feature importance - XGBoost")
ax.set_xlabel("Importance score")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgboost_feature_importance.png", dpi=180)
plt.show()

Hình 3. Tầm quan trọng của 15 đặc trưng đóng góp nhiều nhất trong XGBoost.